# 🚀 CAR-IMU: High-Performance CUDA Evaluation
This notebook is optimized for running 192-d LOSO cross-validation on A100/T4 GPUs using `evaluate_cuda.py`.

In [1]:
# 1. Setup Environment
# Update this line in your Colab Setup cell:
!git clone -b axis-mean-pooling https://github.com/aviral23032002/bio-pm-guided-synthetic-imu-generation.git
%cd bio-pm-guided-synthetic-imu-generation
!pip install h5py torch scikit-learn pandas matplotlib

Cloning into 'bio-pm-guided-synthetic-imu-generation'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 184 (delta 55), reused 166 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 9.72 MiB | 6.22 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/bio-pm-guided-synthetic-imu-generation


In [3]:
# 2. Mount Google Drive & Set Path
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.environ['BIOPM_PATH'] = '/content/drive/MyDrive/BIOPM'
biopm_root = os.environ['BIOPM_PATH']
print(f"🌍 Global BIOPM path: {biopm_root}")

Mounted at /content/drive
🌍 Global BIOPM path: /content/drive/MyDrive/BIOPM


In [4]:
# 3. Handle Data Loading (Unzip to local for max speed)
zip_path = os.path.join(biopm_root, 'car_imu_results_package.zip')

if os.path.exists(zip_path):
    print(f"📦 Unzipping to local runtime...")
    !unzip -q -o "{zip_path}" -d .
    
    # 4. Recover missing token_store.hdf5 from Drive if needed
    local_token_store = "results_wisdm_v2_6class/token_store.hdf5"
    drive_token_store = os.path.join(biopm_root, local_token_store)
    
    if not os.path.exists(local_token_store):
        if os.path.exists(drive_token_store):
            print("🔄 token_store.hdf5 missing from zip. Copying from Drive...")
            !mkdir -p results_wisdm_v2_6class
            !cp "{drive_token_store}" "{local_token_store}"
            print("✅ token_store.hdf5 recovered.")
        else:
            print(f"❌ ERROR: token_store.hdf5 not found on Drive at: {drive_token_store}")
    else:
        print("✅ token_store.hdf5 verified locally.")
    
    print("✨ Setup Ready!")
else:
    print("❌ ERROR: Zip not found on Drive. Please upload car_imu_results_package.zip to BIOPM folder.")

📦 Unzipping to local runtime...
🔄 token_store.hdf5 missing from zip. Copying from Drive...
✅ token_store.hdf5 recovered.
✨ Setup Ready!


In [ ]:

# Run Evaluation
!python evaluate_cuda.py \
    --real "results_wisdm_v2_6class/token_store.hdf5" \
    --syn "synthetic_tokens_v2_6class/synthetic_tokens.hdf5"


/content/bio-pm-guided-synthetic-imu-generation
From https://github.com/aviral23032002/bio-pm-guided-synthetic-imu-generation
 * branch            wisdm2-data-branch -> FETCH_HEAD
HEAD is now at 6888a66 fix: remove redundant pin_memory=True for GPU tensors
CAR-IMU — High Performance CUDA Evaluation
Device: cuda

Loading HDF5 data...
Traceback (most recent call last):
  File "/content/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 171, in <module>
    main()
  File "/content/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 113, in main
    r_feats, r_labels, r_pids, s_feats, s_labels = load_data(args.real, args.syn)
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 76, in load_data
    with h5py.File(real_path, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/h5py/_hl/files.py", line 555, in __init__
  

In [ ]:
import os
import subprocess

# 1. Automatically find the files
def find_file(name):
    try:
        res = subprocess.check_output(['find', '/content', '-name', name]).decode().strip().split('\n')[0]
        return res if res else None
    except:
        return None

real_path = find_file('token_store.hdf5')
syn_path  = find_file('synthetic_tokens.hdf5')

# 2. Run Evaluation if found
if real_path and syn_path:
    print(f"✅ Found Real: {real_path}")
    print(f"✅ Found Syn:  {syn_path}")
    print("\n🚀 Starting High Performance Evaluation...")
    !python evaluate_cuda.py --real "{real_path}" --syn "{syn_path}"
else:
    print("❌ ERROR: Could not find the .hdf5 files.")
    print("Please make sure you have unzipped your results package successfully.")
    if not real_path: print("Missing: token_store.hdf5")
    if not syn_path:  print("Missing: synthetic_tokens.hdf5")


In [14]:
# Use absolute paths to be safe
!python evaluate_cuda.py \
    --real "/content/bio-pm-guided-synthetic-imu-generation/results_wisdm_v2_6class/token_store.hdf5" \
    --syn "/content/bio-pm-guided-synthetic-imu-generation/synthetic_tokens_v2_6class/synthetic_tokens.hdf5"


CAR-IMU — High Performance CUDA Evaluation
Device: cuda

Loading HDF5 data...

🚀 Running LOSO (51 folds)...
Fold  Subj      A_MLP    B_MLP
-----------------------------------
0     1600      0.858    0.841
1     1601      0.755    0.824
2     1602      0.789    0.864
3     1603      0.740    0.646
4     1604      0.801    0.783
5     1605      0.806    0.798
6     1606      0.873    0.783
7     1607      0.670    0.612
8     1608      0.779    0.682
9     1609      0.702    0.718
10    1610      0.857    0.788
11    1611      0.882    0.877
12    1612      0.832    0.830
13    1613      0.657    0.621
14    1614      0.780    0.627
15    1615      0.761    0.743
16    1616      0.898    0.770
17    1617      0.780    0.755
18    1618      0.864    0.804
19    1619      0.897    0.822
20    1620      0.668    0.599
21    1621      0.674    0.639
22    1622      0.628    0.656
23    1623      0.944    0.832
24    1624      0.626    0.567
25    1625      0.949    0.930
Traceback (most rec

In [11]:
!find /content -name "token_store.hdf5"

^C


In [7]:
# 4. Run High-Speed CUDA Evaluation (192-d MLP)
real_tokens = "results_wisdm_v2_6class/token_store.hdf5"
syn_tokens  = "synthetic_tokens_v2_6class/synthetic_tokens.hdf5"

if os.path.exists(real_tokens):
    !python evaluate_cuda.py \
        --real "{real_tokens}" \
        --syn "{syn_tokens}"
else:
    print("❌ MISSING TOKENS. Check the folder structure in your zip.")

CAR-IMU — High Performance CUDA Evaluation
Device: cuda

Loading HDF5 data...

🚀 Running LOSO (51 folds)...
Fold  Subj      A_MLP    B_MLP
-----------------------------------
Traceback (most recent call last):
  File "/content/bio-pm-guided-synthetic-imu-generation/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 171, in <module>
    main()
  File "/content/bio-pm-guided-synthetic-imu-generation/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 141, in main
    pred_a = train_model(X_tr_r_sc, y_tr_r, X_te_sc, 192, DEVICE)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/bio-pm-guided-synthetic-imu-generation/bio-pm-guided-synthetic-imu-generation/evaluate_cuda.py", line 48, in train_model
    for Xb, yb in loader:
                  ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python

In [ ]:
# 5. Save results to Drive
!mkdir -p "$BIOPM_PATH/results_cuda_final"
!cp -r results_local/* "$BIOPM_PATH/results_cuda_final/"